In [1]:
import numpy as np 
import matplotlib.pyplot as plt
import scipy.ndimage as ndi 
import skimage.io as io 
import skimage.data as data # libreria di immagini utili

path = 'C:\\Users\\rocco\\Documents\\università\\ESM\\laboratorio\\Immagini'

## Descrittori locali per la classificazione di immagini

I descrittori locali (_handcrafted features_) sono utili per classificare immagini tramite tessiture. 
L'obiettivo di un descrittore è quello di rappresentare sinteticamente e in maniera robusta le caratteristiche salienti di un'immagine al fine di discriminarla da altre. I descrittori locali descrivono il comportamento dell'immagine in un intorno del pixel (_patch_) e possono essere:
- **Densi**: calcolati per ogni pixel dell'immagine;
- **Sparsi**: calcolati solo per alcuni pixel salienti.

Per ottenere una rappresentazione di tutta l’immagine, l’informazione estratta da tali descrittori viene aggregata ad esempio calcolando l’istogramma. Il **local binary pattern** (LBP) è un descrittore locale denso utilizzato principalmente per l'analisi di texture dell'immagine. LBP è calcolato per ogni pixel effettuando i seguenti passi:
1. Individuazione di un vicinato costituito da P pixel rispetto al pixel centrale;
2. Confronto di ogni pixel del vicinato con il pixel centrale, ottenendo una stringa di P bit dove 1 indica che il pixel del vicinato è >= del pixel centrale, 0 altrimenti.
3. Conversione della stringa di bit in un numero intero da 0 a _(2^P) - 1_.

Il vicinato può avere diverse forme (circolare, quadrato) e diversi raggi. Il confronto del vicinato con il pixel centrale permette di individuare diversi pattern. 



In [ ]:
# Local Binary Pattern 

x = np.float32(data.brick())

h0 = np.array([[1,0,0],[0,-1,0],[0,0,0]], dtype=np.float32) # Nord Ovest
h1 = np.array([[0,1,0],[0,-1,0],[0,0,0]], dtype=np.float32) # Nord
h2 = np.array([[0,0,1],[0,-1,0],[0,0,0]], dtype=np.float32) # Nord Est
h3 = np.array([[0,0,0],[0,-1,1],[0,0,0]], dtype=np.float32) # Est
h4 = np.array([[0,0,0],[0,-1,0],[0,0,1]], dtype=np.float32) # Sud Est
h5 = np.array([[0,0,0],[0,-1,0],[0,1,0]], dtype=np.float32) # Sud  
h6 = np.array([[0,0,0],[0,-1,0],[1,0,0]], dtype=np.float32) # Sud Ovest
h7 = np.array([[0,0,0],[1,-1,0],[0,0,0]], dtype=np.float32) # Ovest

b0 = ndi.correlate(x, h0) >= 0
b1 = ndi.correlate(x, h1) >= 0
b2 = ndi.correlate(x, h2) >= 0
b3 = ndi.correlate(x, h3) >= 0
b4 = ndi.correlate(x, h4) >= 0
b5 = ndi.correlate(x, h5) >= 0
b6 = ndi.correlate(x, h6) >= 0
b7 = ndi.correlate(x, h7) >= 0

y = b0 + b1*2 + b2*4 + b3*8 + b4*16 + b5*32 + b6*64 + b7*128

mappa_uniformi = (y==0) | (y==255)

plt.figure()
plt.imshow(x, clim=[0,255], cmap='gray')
plt.title('input')

plt.figure()
plt.imshow(y, clim=[0,255], cmap='gray')
plt.title('immagine LBP')

plt.figure()
plt.imshow(mappa_uniformi, clim=[0,1], cmap='gray')
plt.title('mappa delle patch uniformi')

## Classificazione tramite apprendimento automatico 
In questa sezione vedremo come usare un descrittore locale per analizzare immagini al microscopio di tessuti tumorali. L'obittivo è classificare le immagini in tumori benigni e maligni. 

In [ ]:
# classificazione SVM

'''
L'istogramma estratto da LBP rappresenta il vettore di features che sarà utilizzato dal classificatore per individuare la migliore partizione dello spazio delle features. In questo esperimento useremo un classificatore lineare SVM.
'''

train_label = np.load(path+'train_label.npy') 
'''
Vettore di etichette per 896 immagini di tessuti tumorali. L'etichetta è 1 se l'immagine è relativa ad un tumore maligno ed è 0 se è relativo ad un tumore benigno.
'''

train_feat = np.load(path+'train_lbp_8_1_default.npy')
'''
Matrice 896x256 contenente i vettori di features, uno per ogni riga.
'''

# Normalizzazione
mu = np.mean(train_feat, 0) #vettore delle medie calcolato sulle righe
sigma = np.std(train_feat, 0) #vettore delle deviazioni std calcolato sulle righe 
train_feat = (train_feat-mu)/(sigma + 1e-15) 

classifier = LinearSVC().fit(train_feat, train_label) # fase di training 

x = io.imread(path+'test_breakhis.png')
y = local_binary_pattern(x,8,1)
feat, b = np.histogram(y.flatten(), bins=np.arange(0,257), density = True)

feat = (feat-mu)/(sigma+1e-15)
feat = np.reshape(feat, (1,-1))

predizione = classifier.predict(feat)
if predizione == 1:
    print('tumore benigno')
else:
    print('tumore maligno')